<a href="https://colab.research.google.com/github/k8peevey-sys/ds2002-fa26/blob/main/Copy_of_2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
before = len(df)
#Droping duplicates
df = df.drop_duplicates()
#Putting into Log
log(1, 'Dropped duplicates', before - len(df))

[1] Dropped duplicates (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
#First I'm going to convert everything to strings
df['price'] = df['price'].astype(str)

#Then I'm going to remove the $
df['price'] = df['price'].str.replace('$', '', regex=False)

#Then I'm going to remove the commas
df['price'] = df['price'].str.replace(',', '')
#Now I'm going to change to floats and record
df['price'] = df['price'].astype(float)
log(2, 'Cleaned price', len(df))

[2] Cleaned price (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
#First I'm gonna convert qty to numeric
df['qty']= pd.to_numeric(df['qty'], errors='coerce')
before = len(df)
#Now I'm going to remove rows where qty is NaN
df = df.dropna(subset=['qty'])
#Now I'm going to remove rows where qty is negative
df = df[df['qty'] > 0]
log(3, 'Cleaned qty', before - len(df))

[3] Cleaned qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [7]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
#First I'm going to look at the variants
print(df['item'].value_counts())
#Then I'm going to build the mapping dict
ITEM_MAP = {"Cheeseburger": "Cheeseburger", "cheese burger": "Cheeseburger", "Foam Finger": "Foam Finger", "foam finger": "Foam Finger", "Rain Poncho": "Rain Poncho", "rain poncho": "Rain Poncho"}

#Counting unique labels
before_uniq = df['item'].nunique()
#Replace variations
df['item'] = df['item'].replace(ITEM_MAP)
#Counting unique labels
after_uniq = df['item'].nunique()

log(4, 'Collapsed item variants into real names', after_uniq)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[4] Collapsed item variants to into real names (3 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [19]:
# TODO
print(df['category'].value_counts())
#Create dictionary
CATEGORY_MAP = {
"Food": "Food",
"food": "Food",
"Merch": "Merch",
"Apparel": "Merch", # business decision
"RainGear": "RainGear",
"rain-gear": "RainGear",
}
before_uniq = df['category'].nunique()
#Apply the mapping
df['category']= df['category'].replace(CAT_MAP)
#Counting unique
after_uniq = df['category'].nunique()
log(5, 'Collapsed category variants into real names', after_uniq)

category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64
[5] Collapsed category variants into real names (3 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [14]:
CATEGORY_MAP = {
    "Food": "Food",
    "food": "Food",
    "Merch": "Merch",
    "Apparel": "Merch", # business decision
    "RainGear": "RainGear",
    "rain-gear": "RainGear",
}

df['category'] = df['category'].replace(CATEGORY_MAP)

assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category'].unique()) == { 'Food', 'Merch', 'RainGear'}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [17]:
# TODO
#First I'm going to create a revenue column
df['revenue'] = df['qty']*df['price']
#Then I'm going to group by category and add the total revenue within each category
rev_by_cat = (df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2))
print('Revenue by category:', rev_by_cat)
#Here I'm calaculating the total revenue
total_revenue = round(df['revenue'].sum(), 2)
print('Total revenue:', total_revenue)
#Here I'm printing the highest category
top_cat = rev_by_cat.index[0]
print('Top category:', top_cat)

Revenue by category: category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Total revenue: 4740.0
Top category: Food


**What I would tell the vendor:** _I would tell the vendor to put more funding into expanding the food in order to make more revenue since it is the highest yield category.

---



### TODO 8 — read back your log

In [18]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,1,Dropped duplicates,15
1,2,Cleaned price,300
2,3,Cleaned qty,25
3,4,Collapsed item variants to into real names,3
4,5,Collapsed category variants to into real names,4
5,5,Collapsed category variants to into real names,4


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) When I removed the missing and negative rows in step 3, the revenue changed from $4595.50 to $4740.00.

b) A reasonable person might not have chosen to combine Apparel and Merch. You could keep them seperate and it wouldn't have changed the revenue. I did it because it makes it easier to read.
